# 07. Evaluación y Análisis de Resultados

Notebook de cierre para comparar modelos, revisar cobertura y dejar una lectura clara de fortalezas y riesgos del sistema.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (10, 5)

from src.utils.recsys_utils import ensure_dir

processed_dir = ROOT / 'data' / 'processed'
tables_dir = ensure_dir(ROOT / 'reports' / 'tables')
models_dir = ensure_dir(ROOT / 'models')

train = pd.read_parquet(processed_dir / 'interactions_train.parquet')
test = pd.read_parquet(processed_dir / 'interactions_test.parquet')


## Consolidar métricas de modelos

Leemos los resultados producidos por notebooks previos y armamos una sola tabla comparativa.


In [ ]:
metric_files = {
    'cf': tables_dir / 'cf_metrics.csv',
    'bandit': tables_dir / 'bandit_metrics.csv',
    'hybrid': tables_dir / 'hybrid_metrics.csv',
}

metrics_frames = []
for name, path in metric_files.items():
    if path.exists():
        frame = pd.read_csv(path)
        frame['source_file'] = path.name
        metrics_frames.append(frame)

if metrics_frames:
    comparison = pd.concat(metrics_frames, ignore_index=True, sort=False)
else:
    comparison = pd.DataFrame()

display(comparison)
comparison.to_csv(tables_dir / 'model_comparison.csv', index=False)


## Segmentación de usuarios e ítems

Analizamos qué segmentos dominan el set de evaluación para contextualizar las métricas.


In [ ]:
user_segment = (
    train.groupby('userId').size()
    .rename('n_ratings')
    .reset_index()
    .assign(segment=lambda df: pd.cut(df['n_ratings'], bins=[0, 50, 100, 250, df['n_ratings'].max()], include_lowest=True))
)

item_segment = (
    train.groupby('movieId').size()
    .rename('n_ratings')
    .reset_index()
    .assign(segment=lambda df: pd.qcut(df['n_ratings'], q=4, duplicates='drop'))
)

display(user_segment.groupby('segment').size().rename('users').reset_index())
display(item_segment.groupby('segment').size().rename('items').reset_index())


## Inspección cualitativa de recomendaciones

Si existen artefactos de recomendaciones, mostramos ejemplos para revisar diversidad y coherencia.


In [ ]:
qualitative_frames = []
for name, file_name in [('cf', 'cf_recommendations.parquet'), ('hybrid', 'hybrid_recommendations.parquet')]:
    path = models_dir / file_name
    if path.exists():
        frame = pd.read_parquet(path).copy()
        frame['model'] = name
        qualitative_frames.append(frame.head(10))

if qualitative_frames:
    qualitative = pd.concat(qualitative_frames, ignore_index=True)
    display(qualitative)
else:
    print('Aun no existen recomendaciones guardadas. Ejecuta primero los notebooks 05a y 05c.')


## Conclusiones finales

Este bloque resume resultados y deja una base para el informe técnico.


In [ ]:
conclusions = pd.DataFrame([
    {'point': 'El baseline colaborativo sirve como referencia fuerte para el catalogo caliente.', 'status': 'esperado'},
    {'point': 'El hibrido deberia mejorar cobertura y cold-start relativo cuando las features de contenido son ricas.', 'status': 'esperado'},
    {'point': 'El experimento bandit es comparativo offline y no reemplaza una evaluacion online.', 'status': 'advertencia'},
])
display(conclusions)
conclusions.to_csv(tables_dir / 'final_conclusions.csv', index=False)
